<a href="https://colab.research.google.com/github/KaptainKris92/HuggingFace_RL_Course/blob/main/notebooks/unit1/LunarLander_v3_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LunarLander PPO experiments

This notebook provides a reusable workflow for training, evaluating, comparing and publishing PPO agents for `LunarLander-v3`.

## Core concepts

The environment gives the agent an eight-value observation describing the lander's position, velocity, angle, angular velocity and leg contact. The agent chooses one of four engine actions.

PPO uses an **actor–critic** architecture:

- The **actor** (`pi`) produces a probability distribution over actions.
- The **critic** (`vf`) estimates the expected future return from the current state.
- PPO uses the critic to estimate whether an action performed better or worse than expected, then updates the actor while clipping excessively large policy changes.

Important parameters:

- `gamma`: how strongly future rewards contribute to the return.
- `gae_lambda`: the bias–variance and temporal-credit-assignment trade-off in advantage estimation. It is not the exploration/exploitation setting.
- `ent_coef`: encourages exploration by preventing the actor from becoming deterministic too early.
- `net_arch`: controls the hidden-layer sizes of the actor and critic.

Models are compared on the same fixed evaluation seeds. By default, a candidate replaces the current Hugging Face model only when its fixed-seed mean reward is higher.

#### Installs and imports

In [2]:
!apt-get update -qq
!apt-get install -y -qq swig cmake ffmpeg xvfb libgl1

%pip install -q \
    "stable-baselines3[extra]==2.9.0" \
    "gymnasium[box2d]==1.3.0" \
    "huggingface-hub<2" \
    pyvirtualdisplay

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package swig4.0.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../swig4.0_4.0.2-1ubuntu1_amd64.deb ...
Unpacking swig4.0 (4.0.2-1ubuntu1) ...
Selecting previously unselected package swig.
Preparing to unpack .../swig_4.0.2-1ubuntu1_all.deb ...
Unpacking swig (4.0.2-1ubuntu1) ...
Setting up swig4.0 (4.0.2-1ubuntu1) ...
Setting up swig (4.0.2-1ubuntu1) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 72.6 MB/s eta 0:00:00


In [3]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Iterable, Sequence
import json
import os
import shutil
import textwrap

import gymnasium as gym
import numpy as np
import pandas as pd

from huggingface_hub import (
    HfApi,
    hf_hub_download,
    notebook_login,
)
from huggingface_hub.utils import (
    EntryNotFoundError,
    RepositoryNotFoundError,
)

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import (
    CheckpointCallback,
    EvalCallback,
)
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor

from pyvirtualdisplay import Display
from IPython.display import Video, display


if not os.environ.get("DISPLAY"):
    virtual_display = Display(
        visible=False,
        size=(1400, 900),
    )
    virtual_display.start()

    print(
        "Virtual display started:",
        virtual_display.is_alive(),
    )
else:
    print("Using existing display:", os.environ["DISPLAY"])

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Virtual display started: True


#### Model loading, evaluation, and comparison functions

In [4]:
def load_ppo_model(
    model_or_path: PPO | str | Path,
    *,
    device: str = "cpu",
) -> PPO:
    """Return an existing PPO object or load one from disk."""

    if isinstance(model_or_path, PPO):
        return model_or_path

    model_path = Path(model_or_path)

    if not model_path.exists():
        raise FileNotFoundError(
            f"Model not found: {model_path}"
        )

    return PPO.load(
        str(model_path),
        device=device,
    )


def normalise_action(action: Any) -> Any:
    """
    Convert a one-element discrete-action array into an integer.

    PPO commonly returns array([2]), while a non-vectorised
    LunarLander environment expects the integer 2.
    """

    action_array = np.asarray(action)

    if action_array.size == 1:
        return int(action_array.item())

    return action


def evaluate_model(
    model_or_path: PPO | str | Path,
    *,
    seeds: Iterable[int],
    env_id: str = "LunarLander-v3",
    env_kwargs: dict[str, Any] | None = None,
    deterministic: bool = True,
    device: str = "cpu",
) -> dict[str, Any]:
    """
    Evaluate a model on an explicit set of episode seeds.

    Explicit seeds make comparisons between models reproducible
    and ensure they face the same landing scenarios.
    """

    model = load_ppo_model(
        model_or_path,
        device=device,
    )

    env_kwargs = dict(env_kwargs or {})
    seed_list = [int(seed) for seed in seeds]

    if not seed_list:
        raise ValueError(
            "At least one evaluation seed is required."
        )

    episode_rewards: list[float] = []
    episode_lengths: list[int] = []

    for seed in seed_list:
        env = Monitor(
            gym.make(
                env_id,
                **env_kwargs,
            )
        )

        try:
            observation, info = env.reset(seed=seed)

            terminated = False
            truncated = False
            total_reward = 0.0
            episode_length = 0

            while not (terminated or truncated):
                action, _ = model.predict(
                    observation,
                    deterministic=deterministic,
                )

                (
                    observation,
                    reward,
                    terminated,
                    truncated,
                    info,
                ) = env.step(
                    normalise_action(action)
                )

                total_reward += float(reward)
                episode_length += 1

            episode_rewards.append(total_reward)
            episode_lengths.append(episode_length)

        finally:
            env.close()

    rewards = np.asarray(
        episode_rewards,
        dtype=float,
    )
    lengths = np.asarray(
        episode_lengths,
        dtype=float,
    )

    mean_reward = float(rewards.mean())
    std_reward = float(rewards.std())

    return {
        "mean_reward": mean_reward,
        "std_reward": std_reward,
        "course_score": mean_reward - std_reward,
        "success_rate_200": float(
            np.mean(rewards >= 200.0)
        ),
        "mean_episode_length": float(
            lengths.mean()
        ),
        "min_reward": float(rewards.min()),
        "max_reward": float(rewards.max()),
        "n_episodes": len(seed_list),
        "rewards": rewards.tolist(),
        "episode_lengths": lengths.astype(int).tolist(),
    }


def metrics_only(
    results: dict[str, Any],
) -> dict[str, float | int]:
    """Return only summary metrics, excluding episode arrays."""

    return {
        "mean_reward": results["mean_reward"],
        "std_reward": results["std_reward"],
        "course_score": results["course_score"],
        "success_rate_200": results["success_rate_200"],
        "mean_episode_length": (
            results["mean_episode_length"]
        ),
        "min_reward": results["min_reward"],
        "max_reward": results["max_reward"],
        "n_episodes": results["n_episodes"],
    }


def compare_candidate_to_hub(
    candidate_model_or_path: PPO | str | Path,
    *,
    repo_id: str,
    hub_model_filename: str,
    evaluation_seeds: Iterable[int],
    env_id: str = "LunarLander-v3",
    env_kwargs: dict[str, Any] | None = None,
    selection_metric: str = "mean_reward",
    min_improvement: float = 0.0,
    device: str = "cpu",
) -> dict[str, Any]:
    """
    Compare a candidate with the current Hub model on the same seeds.

    Valid selection metrics:
    - mean_reward
    - course_score
    - success_rate_200
    """

    valid_metrics = {
        "mean_reward",
        "course_score",
        "success_rate_200",
    }

    if selection_metric not in valid_metrics:
        raise ValueError(
            "selection_metric must be one of "
            f"{sorted(valid_metrics)}."
        )

    seed_list = [
        int(seed)
        for seed in evaluation_seeds
    ]

    candidate_results = evaluate_model(
        candidate_model_or_path,
        seeds=seed_list,
        env_id=env_id,
        env_kwargs=env_kwargs,
        device=device,
    )

    current_results = None
    current_model_path = None

    try:
        current_model_path = hf_hub_download(
            repo_id=repo_id,
            filename=hub_model_filename,
            repo_type="model",
        )

        current_results = evaluate_model(
            current_model_path,
            seeds=seed_list,
            env_id=env_id,
            env_kwargs=env_kwargs,
            device=device,
        )

    except (
        EntryNotFoundError,
        RepositoryNotFoundError,
    ):
        print(
            "No current Hub model was found. "
            "The candidate is eligible for upload."
        )

    rows = [
        {
            "model": "candidate",
            **metrics_only(candidate_results),
        }
    ]

    if current_results is not None:
        rows.append(
            {
                "model": "current_hub",
                **metrics_only(current_results),
            }
        )

    comparison_table = (
        pd.DataFrame(rows)
        .set_index("model")
    )

    if current_results is None:
        improvement = None
        candidate_is_better = True

    else:
        candidate_value = float(
            candidate_results[selection_metric]
        )
        current_value = float(
            current_results[selection_metric]
        )

        improvement = (
            candidate_value - current_value
        )

        candidate_is_better = (
            candidate_value
            > current_value + min_improvement
        )

    display(comparison_table.round(3))

    print("Selection metric:", selection_metric)

    if improvement is not None:
        print(f"Improvement: {improvement:+.3f}")

    print(
        "Candidate is eligible for upload:",
        candidate_is_better,
    )

    return {
        "candidate": candidate_results,
        "current": current_results,
        "current_model_path": current_model_path,
        "selection_metric": selection_metric,
        "min_improvement": min_improvement,
        "improvement": improvement,
        "candidate_is_better": candidate_is_better,
        "table": comparison_table,
    }

#### Train model function

In [5]:
def train_ppo_experiment(
    *,
    experiment_name: str,
    total_timesteps: int,
    env_id: str = "LunarLander-v3",
    env_kwargs: dict[str, Any] | None = None,
    n_envs: int = 16,
    seed: int = 42,
    actor_layers: Sequence[int] = (64, 64),
    critic_layers: Sequence[int] = (64, 64),
    learning_rate: float = 3e-4,
    n_steps: int = 1024,
    batch_size: int = 64,
    n_epochs: int = 4,
    gamma: float = 0.999,
    gae_lambda: float = 0.98,
    ent_coef: float = 0.01,
    clip_range: float = 0.2,
    eval_every_timesteps: int = 50_000,
    checkpoint_every_timesteps: int = 100_000,
    n_eval_episodes: int = 20,
    output_root: str | Path = "/content/rl-experiments",
    device: str = "cpu",
    verbose: int = 1,
) -> dict[str, Any]:
    """
    Train a PPO actor–critic model.

    Saves:
    - periodic safety checkpoints;
    - the checkpoint with the best evaluation mean;
    - the final model;
    - the experiment configuration.
    """

    if not experiment_name.strip():
        raise ValueError(
            "experiment_name must not be empty."
        )

    if total_timesteps <= 0:
        raise ValueError(
            "total_timesteps must be positive."
        )

    env_kwargs = dict(env_kwargs or {})

    output_dir = (
        Path(output_root)
        / experiment_name
    )
    best_model_dir = (
        output_dir
        / "best_model"
    )
    checkpoint_dir = (
        output_dir
        / "checkpoints"
    )
    evaluation_dir = (
        output_dir
        / "evaluations"
    )

    for directory in (
        best_model_dir,
        checkpoint_dir,
        evaluation_dir,
    ):
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )

    configuration = {
        "experiment_name": experiment_name,
        "env_id": env_id,
        "env_kwargs": env_kwargs,
        "total_timesteps": total_timesteps,
        "n_envs": n_envs,
        "seed": seed,
        "actor_layers": list(actor_layers),
        "critic_layers": list(critic_layers),
        "learning_rate": learning_rate,
        "n_steps": n_steps,
        "batch_size": batch_size,
        "n_epochs": n_epochs,
        "gamma": gamma,
        "gae_lambda": gae_lambda,
        "ent_coef": ent_coef,
        "clip_range": clip_range,
        "eval_every_timesteps": (
            eval_every_timesteps
        ),
        "checkpoint_every_timesteps": (
            checkpoint_every_timesteps
        ),
        "n_eval_episodes": n_eval_episodes,
        "device": device,
    }

    config_path = (
        output_dir
        / "training_config.json"
    )
    config_path.write_text(
        json.dumps(
            configuration,
            indent=2,
        ),
        encoding="utf-8",
    )

    train_env = make_vec_env(
        env_id,
        n_envs=n_envs,
        seed=seed,
        env_kwargs=env_kwargs,
    )

    eval_env = Monitor(
        gym.make(
            env_id,
            **env_kwargs,
        )
    )

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(
            best_model_dir
        ),
        log_path=str(evaluation_dir),
        eval_freq=max(
            eval_every_timesteps // n_envs,
            1,
        ),
        n_eval_episodes=n_eval_episodes,
        deterministic=True,
        render=False,
        verbose=1,
    )

    checkpoint_callback = CheckpointCallback(
        save_freq=max(
            checkpoint_every_timesteps // n_envs,
            1,
        ),
        save_path=str(checkpoint_dir),
        name_prefix=experiment_name,
        verbose=2,
    )

    policy_kwargs = {
        "net_arch": {
            "pi": list(actor_layers),
            "vf": list(critic_layers),
        }
    }

    model = PPO(
        policy="MlpPolicy",
        env=train_env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        batch_size=batch_size,
        n_epochs=n_epochs,
        gamma=gamma,
        gae_lambda=gae_lambda,
        ent_coef=ent_coef,
        clip_range=clip_range,
        policy_kwargs=policy_kwargs,
        device=device,
        seed=seed,
        verbose=verbose,
    )

    try:
        model.learn(
            total_timesteps=total_timesteps,
            callback=[
                eval_callback,
                checkpoint_callback,
            ],
            progress_bar=True,
        )

        final_model_stem = (
            output_dir
            / "final_model"
        )
        model.save(
            str(final_model_stem)
        )

    finally:
        train_env.close()
        eval_env.close()

    best_model_path = (
        best_model_dir
        / "best_model.zip"
    )
    final_model_path = (
        output_dir
        / "final_model.zip"
    )
    evaluations_path = (
        evaluation_dir
        / "evaluations.npz"
    )

    if not best_model_path.exists():
        raise FileNotFoundError(
            "No best_model.zip was created. "
            "Check whether training reached an "
            "evaluation interval."
        )

    print("Best model:", best_model_path)
    print("Final model:", final_model_path)

    return {
        "experiment_name": experiment_name,
        "output_dir": output_dir,
        "best_model_path": best_model_path,
        "final_model_path": final_model_path,
        "evaluations_path": evaluations_path,
        "config_path": config_path,
        "configuration": configuration,
    }

#### Record video function

In [6]:
def record_replay(
    model_or_path: PPO | str | Path,
    *,
    output_path: str | Path,
    env_id: str = "LunarLander-v3",
    env_kwargs: dict[str, Any] | None = None,
    seed: int = 42,
    deterministic: bool = True,
    device: str = "cpu",
) -> dict[str, Any]:
    """Record one complete deterministic episode."""

    model = load_ppo_model(
        model_or_path,
        device=device,
    )

    env_kwargs = dict(env_kwargs or {})
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    recording_dir = (
        output_path.parent
        / "_temporary_recording"
    )

    if recording_dir.exists():
        shutil.rmtree(recording_dir)

    recording_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    env = gym.make(
        env_id,
        render_mode="rgb_array",
        **env_kwargs,
    )

    env = gym.wrappers.RecordVideo(
        env,
        video_folder=str(recording_dir),
        episode_trigger=(
            lambda episode_number:
            episode_number == 0
        ),
        video_length=0,
        name_prefix=output_path.stem,
        disable_logger=True,
    )

    total_reward = 0.0
    steps = 0

    try:
        observation, info = env.reset(
            seed=seed
        )

        terminated = False
        truncated = False

        while not (terminated or truncated):
            action, _ = model.predict(
                observation,
                deterministic=deterministic,
            )

            (
                observation,
                reward,
                terminated,
                truncated,
                info,
            ) = env.step(
                normalise_action(action)
            )

            total_reward += float(reward)
            steps += 1

    finally:
        env.close()

    generated_videos = list(
        recording_dir.glob("*.mp4")
    )

    if not generated_videos:
        raise FileNotFoundError(
            "No MP4 replay was generated."
        )

    generated_video = max(
        generated_videos,
        key=lambda path:
        path.stat().st_mtime,
    )

    shutil.copy2(
        generated_video,
        output_path,
    )

    shutil.rmtree(
        recording_dir,
        ignore_errors=True,
    )

    print(
        f"Replay saved to {output_path}"
    )
    print(
        f"Replay reward: {total_reward:.2f}"
    )
    print(
        f"Replay length: {steps} steps"
    )

    return {
        "path": str(output_path),
        "seed": seed,
        "reward": total_reward,
        "steps": steps,
    }

#### Model-card and Hugging Face repo upload functions

In [7]:
def build_model_card(
    *,
    repo_id: str,
    env_id: str,
    model_filename: str,
    evaluation: dict[str, Any],
    replay: dict[str, Any],
    training_config: dict[str, Any] | None,
    comparison: dict[str, Any] | None,
) -> str:
    """Generate the repository README/model card."""

    actor_layers = (
        training_config.get("actor_layers")
        if training_config
        else "Not supplied"
    )
    critic_layers = (
        training_config.get("critic_layers")
        if training_config
        else "Not supplied"
    )

    comparison_text = ""

    if (
        comparison is not None
        and comparison["current"] is not None
    ):
        comparison_text = (
            f"The candidate was compared with the "
            f"previous Hub model on the same "
            f"{evaluation['n_episodes']} fixed seeds. "
            f"The selection metric was "
            f"`{comparison['selection_metric']}` and "
            f"the observed improvement was "
            f"{comparison['improvement']:+.3f}."
        )

    replay_url = (
        f"https://huggingface.co/"
        f"{repo_id}/resolve/main/replay.mp4"
    )

    return textwrap.dedent(
        f"""
        ---
        library_name: stable-baselines3
        pipeline_tag: reinforcement-learning
        tags:
        - stable-baselines3
        - reinforcement-learning
        - deep-reinforcement-learning
        - PPO
        - {env_id}
        ---

        # PPO agent for {env_id}

        This repository contains a Stable-Baselines3 PPO actor–critic agent trained on `{env_id}`.

        ## Evaluation

        Deterministic evaluation over {evaluation['n_episodes']} fixed-seed episodes:

        | Metric | Value |
        |---|---:|
        | Mean reward | {evaluation['mean_reward']:.2f} |
        | Standard deviation | {evaluation['std_reward']:.2f} |
        | Course-style score (`mean - std`) | {evaluation['course_score']:.2f} |
        | Episodes scoring at least 200 | {evaluation['success_rate_200']:.1%} |
        | Minimum reward | {evaluation['min_reward']:.2f} |
        | Maximum reward | {evaluation['max_reward']:.2f} |

        {comparison_text}

        ## Architecture

        - Algorithm: PPO
        - Policy: MLP actor–critic
        - Actor hidden layers: `{actor_layers}`
        - Critic hidden layers: `{critic_layers}`

        ## Replay

        Replay seed: `{replay['seed']}`
        Replay reward: `{replay['reward']:.2f}`

        <video controls autoplay loop muted width="640">
          <source src="{replay_url}" type="video/mp4">
        </video>

        ## Load the model

        ```python
        from huggingface_hub import hf_hub_download
        from stable_baselines3 import PPO

        checkpoint = hf_hub_download(
            repo_id="{repo_id}",
            filename="{model_filename}",
        )

        model = PPO.load(checkpoint)
        ```
        """
    ).strip()


def promote_model_to_hub(
    candidate_model_or_path: PPO | str | Path,
    *,
    repo_id: str,
    model_filename: str = "ppo-LunarLander-v3.zip",
    env_id: str = "LunarLander-v3",
    env_kwargs: dict[str, Any] | None = None,
    evaluation_seeds: Iterable[int] = range(
        10_000,
        10_100,
    ),
    compare_to_current: bool = True,
    selection_metric: str = "mean_reward",
    min_improvement: float = 0.0,
    force_upload: bool = False,
    upload: bool = True,
    video_seed: int = 42,
    training_config_path: str | Path | None = None,
    commit_message: str | None = None,
    private: bool = False,
    device: str = "cpu",
) -> dict[str, Any]:
    """
    Evaluate, compare, record and publish a candidate model.

    Parameters
    ----------
    compare_to_current:
        Compare against the model currently stored in the repo.

    selection_metric:
        "mean_reward", "course_score" or "success_rate_200".

    min_improvement:
        Required improvement over the current model.

    force_upload:
        Upload even if the candidate does not win.

    upload:
        Set False for a dry run that creates local staging files.
    """

    candidate_model = load_ppo_model(
        candidate_model_or_path,
        device=device,
    )

    seed_list = [
        int(seed)
        for seed in evaluation_seeds
    ]

    comparison = None

    if compare_to_current:
        comparison = compare_candidate_to_hub(
            candidate_model,
            repo_id=repo_id,
            hub_model_filename=model_filename,
            evaluation_seeds=seed_list,
            env_id=env_id,
            env_kwargs=env_kwargs,
            selection_metric=selection_metric,
            min_improvement=min_improvement,
            device=device,
        )

        candidate_results = (
            comparison["candidate"]
        )
        eligible = (
            comparison["candidate_is_better"]
        )

    else:
        candidate_results = evaluate_model(
            candidate_model,
            seeds=seed_list,
            env_id=env_id,
            env_kwargs=env_kwargs,
            device=device,
        )
        eligible = True

    if not eligible and not force_upload:
        print(
            "The candidate did not meet the "
            "promotion criterion. Nothing was uploaded."
        )

        return {
            "uploaded": False,
            "eligible": False,
            "comparison": comparison,
        }

    staging_dir = (
        Path("/content/hf-staging")
        / repo_id.replace("/", "__")
    )

    if staging_dir.exists():
        shutil.rmtree(staging_dir)

    staging_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    model_stem = (
        staging_dir
        / Path(model_filename).stem
    )

    candidate_model.save(
        str(model_stem)
    )

    saved_model_path = (
        staging_dir
        / model_filename
    )

    if not saved_model_path.exists():
        raise FileNotFoundError(
            f"Expected saved model at "
            f"{saved_model_path}"
        )

    replay = record_replay(
        candidate_model,
        output_path=(
            staging_dir
            / "replay.mp4"
        ),
        env_id=env_id,
        env_kwargs=env_kwargs,
        seed=video_seed,
        device=device,
    )

    training_config = None

    if training_config_path is not None:
        training_config_path = Path(
            training_config_path
        )

        if training_config_path.exists():
            training_config = json.loads(
                training_config_path.read_text(
                    encoding="utf-8"
                )
            )

            shutil.copy2(
                training_config_path,
                staging_dir
                / "training_config.json",
            )

    results_payload = {
        "env_id": env_id,
        "env_kwargs": dict(
            env_kwargs or {}
        ),
        "selection_metric": selection_metric,
        "min_improvement": min_improvement,
        "candidate": metrics_only(
            candidate_results
        ),
        "current_hub": (
            metrics_only(
                comparison["current"]
            )
            if comparison is not None
            and comparison["current"] is not None
            else None
        ),
        "improvement": (
            comparison["improvement"]
            if comparison is not None
            else None
        ),
        "evaluation_seeds": seed_list,
        "replay": replay,
    }

    (
        staging_dir
        / "results.json"
    ).write_text(
        json.dumps(
            results_payload,
            indent=2,
        ),
        encoding="utf-8",
    )

    config_payload = {
        "algorithm": "PPO",
        "env_id": env_id,
        "model_name": Path(
            model_filename
        ).stem,
        "mean_reward": (
            candidate_results["mean_reward"]
        ),
        "std_reward": (
            candidate_results["std_reward"]
        ),
    }

    (
        staging_dir
        / "config.json"
    ).write_text(
        json.dumps(
            config_payload,
            indent=2,
        ),
        encoding="utf-8",
    )

    readme = build_model_card(
        repo_id=repo_id,
        env_id=env_id,
        model_filename=model_filename,
        evaluation=candidate_results,
        replay=replay,
        training_config=training_config,
        comparison=comparison,
    )

    (
        staging_dir
        / "README.md"
    ).write_text(
        readme,
        encoding="utf-8",
    )

    print(
        "Staging files:",
        sorted(
            path.name
            for path in staging_dir.iterdir()
        ),
    )

    if not upload:
        print(
            "Dry run complete. Nothing was uploaded."
        )

        return {
            "uploaded": False,
            "eligible": True,
            "comparison": comparison,
            "staging_dir": staging_dir,
        }

    api = HfApi()

    api.create_repo(
        repo_id=repo_id,
        repo_type="model",
        private=private,
        exist_ok=True,
    )

    upload_result = api.upload_folder(
        folder_path=str(staging_dir),
        repo_id=repo_id,
        repo_type="model",
        commit_message=(
            commit_message
            or (
                "Promote improved PPO model "
                f"({selection_metric}="
                f"{candidate_results[selection_metric]:.2f})"
            )
        ),
    )

    print("Uploaded:", upload_result)

    return {
        "uploaded": True,
        "eligible": True,
        "comparison": comparison,
        "staging_dir": staging_dir,
        "upload_result": upload_result,
    }

#### Log in to Hugging Face

In [8]:
notebook_login()

## Model training

### 128 x 128 actor-critic model

In [ ]:
run_128 = train_ppo_experiment(
    experiment_name="ppo_lunarlander_128x128",
    total_timesteps=5_000_000,
    env_id="LunarLander-v3",
    n_envs=16,
    seed=42,

    # Larger actor and critic
    actor_layers=(128, 128),
    critic_layers=(128, 128),

    # Tuned PPO parameters
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.01,
    clip_range=0.2,

    eval_every_timesteps=50_000,
    checkpoint_every_timesteps=100_000,
    n_eval_episodes=20,

    device="cpu",
)

run_128

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


Using cpu device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

...

Saving model checkpoint to 
/content/rl-experiments/ppo_lunarlander_128x128/checkpoints/ppo_lunarlander_128x128_100000_steps.zip

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 90.8     |
|    ep_rew_mean     | -87.7    |
| time/              |          |
|    fps             | 1277     |
|    iterations      | 7        |
|    time_elapsed    | 89       |
|    total_timesteps | 114688   |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 97.7        |
|    ep_rew_mean          | -70.9       |
| time/                   |             |
|    fps                  | 1304        |
|    iterations           | 8           |
|    time_elapsed         | 100         |
|    total_timesteps      | 131072      |
| train/                  |             |
|    approx_kl            | 0.007980425 |
|    clip_fraction        | 0.0739      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.31       |
|    explained_variance   | 0.0226      |
|    learning_rate        | 0.

Eval num_timesteps=150000, episode_reward=-190.21 +/- 41.91

Episode length: 490.00 +/- 209.01

-----------------------------------------
| eval/                   |             |
|    mean_ep_length       | 490         |
|    mean_reward          | -190        |
| time/                   |             |
|    total_timesteps      | 150000      |
| train/                  |             |
|    approx_kl            | 0.009779174 |
|    clip_fraction        | 0.0667      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.25       |
|    explained_variance   | 0.079       |
|    learning_rate        | 0.0003      |
|    loss                 | 152         |
|    n_updates            | 36          |
|    policy_gradient_loss | -0.00658    |
|    value_loss           | 299         |
-----------------------------------------


#### Evaluate new best checkpoint

In [ ]:
EVALUATION_SEEDS = list(
    range(10_000, 10_100)
)

results_128 = evaluate_model(
    run_128["best_model_path"],
    seeds=EVALUATION_SEEDS,
    env_id="LunarLander-v3",
)

pd.Series(
    metrics_only(results_128),
    name="128x128 model",
)

#### Compare with current Hub model

In [ ]:
comparison_128 = compare_candidate_to_hub(
    run_128["best_model_path"],
    repo_id="KaptainKris/HuggingFace_RL_Course",
    hub_model_filename="ppo-LunarLander-v3.zip",
    evaluation_seeds=EVALUATION_SEEDS,
    env_id="LunarLander-v3",

    # Compare by mean reward rather than mean - std
    # Alternative selection metric: "course_score"
    selection_metric="mean_reward",

    # Any positive improvement is sufficient
    min_improvement=0.0,
)

#### Upload only if new model is better

In [ ]:
promotion_128 = promote_model_to_hub(
    run_128["best_model_path"],

    repo_id="KaptainKris/HuggingFace_RL_Course",
    model_filename="ppo-LunarLander-v3.zip",
    env_id="LunarLander-v3",

    evaluation_seeds=EVALUATION_SEEDS,

    compare_to_current=True,
    selection_metric="mean_reward",
    min_improvement=0.0,

    # Leave False to protect the current model
    force_upload=False,

    # Set False to create files without uploading
    upload=True,

    video_seed=42,
    training_config_path=run_128["config_path"],

    commit_message=(
        "Promote 128x128 PPO LunarLander-v3 model"
    ),
)

promotion_128

#### Preview generated video locally

In [ ]:
replay_path = (
    promotion_128["staging_dir"]
    / "replay.mp4"
)

display(
    Video(
        str(replay_path),
        embed=True,
    )
)

#### Local backup

In [ ]:
from google.colab import files

files.download(
    str(run_128["best_model_path"])
)